# 19 · Cross-Height Generalization

This notebook asks whether the low-dimensional local structure found earlier is **transferable across height**.

## Core question

If we learn a simple separation rule at one zeta-height block, does it still work at a different zeta-height block?

We test three binary tasks:

1. **zeta vs Poisson**
2. **GUE vs Poisson**
3. **zeta vs GUE**

but now in a cross-height setting:
- train on one zeta block
- test on another zeta block

## Why this matters

Earlier notebooks showed:
- geometric similarity between zeta and GUE
- stability across height
- conditional structure across height

This notebook asks a stronger question:

> Is the decision boundary itself stable across height?

If yes, the local descriptor geometry is not just descriptive; it is transferable.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 1800
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Minimal feature set from earlier notebooks

We use the strongest compact feature set:

\[
(\text{entropy},\ \text{unevenness},\ \text{ratio\_mean})
\]

on normalized 5-gap windows.

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def build_feature_matrix(gaps, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    X = np.array([
        [window_entropy(w), unevenness(w), ratio_mean(w)]
        for w in windows_n
    ], dtype=float)
    return windows_n, X

## Simple classifier helpers

We use:
- nearest centroid
- k-nearest neighbors (k=5)

trained on one block and tested on another.

In [ ]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

def nearest_centroid_fit(X_train, y_train):
    classes = np.unique(y_train)
    centroids = {c: X_train[y_train == c].mean(axis=0) for c in classes}
    return centroids

def nearest_centroid_predict(X, centroids):
    classes = sorted(centroids.keys())
    centroid_array = np.vstack([centroids[c] for c in classes])
    dists = np.linalg.norm(X[:, None, :] - centroid_array[None, :, :], axis=2)
    pred_idx = np.argmin(dists, axis=1)
    return np.array([classes[i] for i in pred_idx])

def knn_predict(X_train, y_train, X_test, k=5):
    preds = []
    for x in X_test:
        d = np.linalg.norm(X_train - x, axis=1)
        nn = np.argsort(d)[:k]
        votes = y_train[nn]
        counts = np.bincount(votes, minlength=2)
        preds.append(np.argmax(counts))
    return np.array(preds)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def confusion_2x2(y_true, y_pred):
    return np.array([
        [np.sum((y_true == 0) & (y_pred == 0)), np.sum((y_true == 0) & (y_pred == 1))],
        [np.sum((y_true == 1) & (y_pred == 0)), np.sum((y_true == 1) & (y_pred == 1))]
    ])

## Height blocks

In [ ]:
block_size = 300
block_starts = [0, 300, 600, 900, 1200]
block_labels = [f"{s+1}-{s+block_size}" for s in block_starts]
block_labels

## Build blockwise feature matrices

In [ ]:
zeta_blocks = {}
poisson_blocks = {}

for s, label in zip(block_starts, block_labels):
    zeta_block = zeta_gaps_full[s:s + block_size]
    poisson_block = poisson_gaps_full[s:s + block_size]

    _, zeta_X = build_feature_matrix(zeta_block, k=5)
    _, poisson_X = build_feature_matrix(poisson_block, k=5)

    zeta_blocks[label] = zeta_X
    poisson_blocks[label] = poisson_X

_, gue_X_full = build_feature_matrix(gue_gaps_full[:1200], k=5)

{label: zeta_blocks[label].shape for label in block_labels}, gue_X_full.shape

## Cross-height task runner

In [ ]:
def run_cross_height_task(name, X0_train, X1_train, X0_test, X1_test):
    m_train = min(len(X0_train), len(X1_train))
    m_test = min(len(X0_test), len(X1_test))

    X_train = np.vstack([X0_train[:m_train], X1_train[:m_train]])
    y_train = np.array([0] * m_train + [1] * m_train)

    X_test = np.vstack([X0_test[:m_test], X1_test[:m_test]])
    y_test = np.array([0] * m_test + [1] * m_test)

    X_train_s, X_test_s = standardize_train_test(X_train, X_test)

    centroids = nearest_centroid_fit(X_train_s, y_train)
    y_pred_centroid = nearest_centroid_predict(X_test_s, centroids)
    y_pred_knn = knn_predict(X_train_s, y_train, X_test_s, k=5)

    return {
        "task": name,
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "centroid_accuracy": accuracy(y_test, y_pred_centroid),
        "knn_accuracy": accuracy(y_test, y_pred_knn),
        "centroid_confusion": confusion_2x2(y_test, y_pred_centroid).tolist(),
        "knn_confusion": confusion_2x2(y_test, y_pred_knn).tolist(),
    }

## Main cross-height experiments

We train on the first block and test on later blocks.

Train block:
- 1–300

Test blocks:
- 301–600
- 601–900
- 901–1200
- 1201–1500

In [ ]:
train_label = block_labels[0]
test_labels = block_labels[1:]

results = []

for test_label in test_labels:
    zeta_train = zeta_blocks[train_label]
    zeta_test = zeta_blocks[test_label]

    pois_train = poisson_blocks[train_label]
    pois_test = poisson_blocks[test_label]

    m_train_gue = min(len(gue_X_full), len(zeta_train), len(pois_train))
    m_test_gue = min(len(gue_X_full), len(zeta_test), len(pois_test))

    gue_train = gue_X_full[:m_train_gue]
    gue_test = gue_X_full[-m_test_gue:]

    results.append({
        "train_block": train_label,
        "test_block": test_label,
        "zeta_vs_Poisson": run_cross_height_task(
            "zeta vs Poisson",
            zeta_train, pois_train,
            zeta_test, pois_test
        ),
        "GUE_vs_Poisson": run_cross_height_task(
            "GUE vs Poisson",
            gue_train, pois_train,
            gue_test, pois_test
        ),
        "zeta_vs_GUE": run_cross_height_task(
            "zeta vs GUE",
            zeta_train, gue_train,
            zeta_test, gue_test
        ),
    })

results

## Accuracy summary across test heights

In [ ]:
x = np.arange(len(test_labels))
width = 0.18

tasks = ["zeta_vs_Poisson", "GUE_vs_Poisson", "zeta_vs_GUE"]
titles = ["zeta vs Poisson", "GUE vs Poisson", "zeta vs GUE"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)

for ax, task_key, title in zip(axes, tasks, titles):
    centroid_vals = [r[task_key]["centroid_accuracy"] for r in results]
    knn_vals = [r[task_key]["knn_accuracy"] for r in results]

    ax.bar(x - width/2, centroid_vals, width, label="nearest centroid")
    ax.bar(x + width/2, knn_vals, width, label="k-NN (k=5)")
    ax.set_xticks(x, test_labels, rotation=25)
    ax.set_ylim(0.4, 1.0)
    ax.set_title(title)
    ax.set_ylabel("accuracy")

axes[0].legend()
plt.tight_layout()
plt.show()

## Full confusion matrices

In [ ]:
for r in results:
    print("\n" + "="*72)
    print(f"train {r['train_block']} → test {r['test_block']}")
    for task_key in ["zeta_vs_Poisson", "GUE_vs_Poisson", "zeta_vs_GUE"]:
        task = r[task_key]
        print(f"\n{task['task']}")
        print("Nearest-centroid confusion:")
        print(np.array(task["centroid_confusion"]))
        print("k-NN confusion:")
        print(np.array(task["knn_confusion"]))

## Reverse-direction check

Train on the last block and test backward on the earlier blocks.

In [ ]:
reverse_train_label = block_labels[-1]
reverse_test_labels = block_labels[:-1]

reverse_results = []

for test_label in reverse_test_labels:
    zeta_train = zeta_blocks[reverse_train_label]
    zeta_test = zeta_blocks[test_label]

    pois_train = poisson_blocks[reverse_train_label]
    pois_test = poisson_blocks[test_label]

    m_train_gue = min(len(gue_X_full), len(zeta_train), len(pois_train))
    m_test_gue = min(len(gue_X_full), len(zeta_test), len(pois_test))

    gue_train = gue_X_full[:m_train_gue]
    gue_test = gue_X_full[-m_test_gue:]

    reverse_results.append({
        "train_block": reverse_train_label,
        "test_block": test_label,
        "zeta_vs_Poisson": run_cross_height_task(
            "zeta vs Poisson",
            zeta_train, pois_train,
            zeta_test, pois_test
        ),
        "GUE_vs_Poisson": run_cross_height_task(
            "GUE vs Poisson",
            gue_train, pois_train,
            gue_test, pois_test
        ),
        "zeta_vs_GUE": run_cross_height_task(
            "zeta vs GUE",
            zeta_train, gue_train,
            zeta_test, gue_test
        ),
    })

reverse_results

## Reverse-direction accuracy summary

In [ ]:
x = np.arange(len(reverse_test_labels))
width = 0.18

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)

for ax, task_key, title in zip(axes, tasks, titles):
    centroid_vals = [r[task_key]["centroid_accuracy"] for r in reverse_results]
    knn_vals = [r[task_key]["knn_accuracy"] for r in reverse_results]

    ax.bar(x - width/2, centroid_vals, width, label="nearest centroid")
    ax.bar(x + width/2, knn_vals, width, label="k-NN (k=5)")
    ax.set_xticks(x, reverse_test_labels, rotation=25)
    ax.set_ylim(0.4, 1.0)
    ax.set_title(title)
    ax.set_ylabel("accuracy")

axes[0].legend()
plt.tight_layout()
plt.show()

## Compact summary table

In [ ]:
summary = {
    "forward_transfer": [
        {
            "train_block": r["train_block"],
            "test_block": r["test_block"],
            "zeta_vs_Poisson_centroid": r["zeta_vs_Poisson"]["centroid_accuracy"],
            "zeta_vs_Poisson_knn": r["zeta_vs_Poisson"]["knn_accuracy"],
            "GUE_vs_Poisson_centroid": r["GUE_vs_Poisson"]["centroid_accuracy"],
            "GUE_vs_Poisson_knn": r["GUE_vs_Poisson"]["knn_accuracy"],
            "zeta_vs_GUE_centroid": r["zeta_vs_GUE"]["centroid_accuracy"],
            "zeta_vs_GUE_knn": r["zeta_vs_GUE"]["knn_accuracy"],
        }
        for r in results
    ],
    "reverse_transfer": [
        {
            "train_block": r["train_block"],
            "test_block": r["test_block"],
            "zeta_vs_Poisson_centroid": r["zeta_vs_Poisson"]["centroid_accuracy"],
            "zeta_vs_Poisson_knn": r["zeta_vs_Poisson"]["knn_accuracy"],
            "GUE_vs_Poisson_centroid": r["GUE_vs_Poisson"]["centroid_accuracy"],
            "GUE_vs_Poisson_knn": r["GUE_vs_Poisson"]["knn_accuracy"],
            "zeta_vs_GUE_centroid": r["zeta_vs_GUE"]["centroid_accuracy"],
            "zeta_vs_GUE_knn": r["zeta_vs_GUE"]["knn_accuracy"],
        }
        for r in reverse_results
    ],
}
summary

## Notes

- If zeta-vs-Poisson and GUE-vs-Poisson remain easy under cross-height transfer, that supports the idea that the descriptor geometry is not just stable but predictive across height.
- If zeta-vs-GUE remains much harder, that reinforces the interpretation that zeta and GUE share the same low-dimensional local structure rather than merely looking similar in a single block.
- This notebook uses one fixed minimal feature set. Later notebooks could compare transfer performance for several alternative feature sets.

## Next directions

A natural next notebook could do one of these:

1. add bootstrap uncertainty for cross-height accuracies  
2. test conditional cross-height classifiers (after small vs after large gaps)  
3. compare several minimal feature sets side by side  
4. test train-on-middle, test-on-both-ends generalization